**TAISS 2026 - Filière F1 (Data Science)**

**TP2 - Segmentation RFM**

**Partie 1 : Nettoyage des transactions brutes**

**Entrée :** `../data/raw/online_retail_II.xlsx`

**Sortie :** `../data/processed/transactions_clean.parquet` — une ligne par ligne de transaction valide,
avec une colonne `Amount` (Quantity × Price), prête pour l'agrégation RFM de la Partie 2.

> Version corrigée : voir les cellules marquées **[CORRECTIF]** pour le détail des points traités
> (codes produit spéciaux, annulations, valeurs aberrantes de prix, export au format attendu par la Partie 2).

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

In [6]:
sheets = pd.read_excel(RAW / "online_retail_II.xlsx", sheet_name=None)

print(f"Feuilles excels chargées : {list(sheets.keys())}")

df = pd.concat(sheets.values(), ignore_index=True)

print("Données prêtes.")
display(df.head())

Feuilles excels chargées : ['Year 2009-2010', 'Year 2010-2011']
Données prêtes.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Étape 1 : Inspection

In [7]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB
None


**[CORRECTIF] Inspection approfondie**

`df.info()` seul ne suffit pas : on regarde aussi la plage de dates, les cardinalités des colonnes
clés, et surtout deux familles de lignes qui ne sont *pas* des ventes normales et qu'il faut traiter
explicitement plus loin — les annulations (`Invoice` commençant par `C`) et les prix nuls/négatifs.

In [8]:
print("Plage de dates :", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print()
print("Cardinalités :")
for c in ["Invoice", "StockCode", "Customer ID", "Country"]:
    print(f"  {c}: {df[c].nunique():,} valeurs uniques")

n_annulations = df["Invoice"].astype(str).str.startswith("C").sum()
n_prix_non_positif = (df["Price"] <= 0).sum()
n_qty_non_positive = (df["Quantity"] <= 0).sum()

print()
print(f"Factures d'annulation (Invoice commençant par 'C') : {n_annulations:,} lignes "
      f"({n_annulations/len(df):.1%})")
print(f"Lignes avec Price <= 0                              : {n_prix_non_positif:,} lignes")
print(f"Lignes avec Quantity <= 0                            : {n_qty_non_positive:,} lignes")

Plage de dates : 2009-12-01 07:45:00 -> 2011-12-09 12:50:00

Cardinalités :
  Invoice: 53,628 valeurs uniques
  StockCode: 5,305 valeurs uniques
  Customer ID: 5,942 valeurs uniques
  Country: 43 valeurs uniques

Factures d'annulation (Invoice commençant par 'C') : 19,494 lignes (1.8%)
Lignes avec Price <= 0                              : 6,207 lignes
Lignes avec Quantity <= 0                            : 22,950 lignes


## Étape 2 : Nettoyage

In [9]:
df = df.rename(columns={"Customer ID": "CustomerID"})
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 1. Doublons exacts

In [10]:
cols = df.columns
nb_doublons = df.duplicated(subset=cols).sum()
print(f"Nombre de doublons exacts : {nb_doublons} ({(nb_doublons * 100 / len(df)):.2f} %)")

Nombre de doublons exacts : 34335 (3.22 %)


In [11]:
print("Aperçu de quelques doublons")
display(df[df.duplicated(subset=cols)].head(20))

Aperçu de quelques doublons


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
657,489529,22028,PENNY FARTHING BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984.0,United Kingdom
658,489529,22036,DINOSAUR BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984.0,United Kingdom


In [12]:
df = df.drop_duplicates(subset=cols).reset_index(drop=True)
print(f"Taille des données après suppression des doublons : {df.shape[0]} lignes")

Taille des données après suppression des doublons : 1033036 lignes


### 2. Valeurs manquantes

In [13]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
CustomerID     235151
Country             0
dtype: int64

In [14]:
df[df["CustomerID"].isna()].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
462,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
569,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
570,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1027,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1028,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1029,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1030,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


**[CORRECTIF]** On vérifie aussi si les `Description` manquantes se recoupent avec les `CustomerID`
manquants (souvent le même type de lignes d'ajustement comptable, pas de vraies ventes).

In [15]:
print("Description manquante :", df["Description"].isna().sum())
print("Chevauchement CustomerID manquant ET Description manquante :",
      (df["CustomerID"].isna() & df["Description"].isna()).sum())

Description manquante : 4275
Chevauchement CustomerID manquant ET Description manquante : 4275


In [16]:
df_cleaned = df.dropna(subset=["CustomerID"]).reset_index(drop=True)
print(f"Taille après suppression des CustomerID manquants : {len(df_cleaned)} lignes")

Taille après suppression des CustomerID manquants : 797885 lignes


In [17]:
print(df_cleaned.isna().sum())

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
CustomerID     0
Country        0
dtype: int64


### 3. Type de données et incohérences métier

In [18]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797885 non-null  object        
 1   StockCode    797885 non-null  object        
 2   Description  797885 non-null  object        
 3   Quantity     797885 non-null  int64         
 4   InvoiceDate  797885 non-null  datetime64[us]
 5   Price        797885 non-null  float64       
 6   CustomerID   797885 non-null  float64       
 7   Country      797885 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 58.9+ MB


In [19]:
df_cleaned["CustomerID"] = df_cleaned["CustomerID"].astype(int)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797885 non-null  object        
 1   StockCode    797885 non-null  object        
 2   Description  797885 non-null  object        
 3   Quantity     797885 non-null  int64         
 4   InvoiceDate  797885 non-null  datetime64[us]
 5   Price        797885 non-null  float64       
 6   CustomerID   797885 non-null  int64         
 7   Country      797885 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), object(3), str(1)
memory usage: 58.9+ MB


**[CORRECTIF] StockCode — ne pas supprimer les codes spéciaux à l'aveugle**

La version précédente extrayait uniquement les chiffres de `StockCode` (`"79323W"` → `79323`) puis
supprimait toute ligne devenue `NaN` — donc **toutes** les lignes dont le code est purement
alphabétique (`POST`, `D`, `M`, `BANK CHARGES`, `DOT`, `C2`, `AMAZONFEE`, cartes cadeaux...)
disparaissaient sans qu'on sache combien ni pourquoi. Ces lignes ne sont pas des erreurs de
saisie : ce sont de vraies charges (frais de port, remise, frais bancaires) qui contribuent au
montant réellement facturé au client.

On commence donc par **diagnostiquer** ces codes avant de décider quoi en faire.

In [20]:
def est_code_numerique(code):
    """Un StockCode 'produit standard' contient au moins un chiffre et,
    une fois les caractères non numériques retirés, redevient un entier."""
    s = str(code)
    chiffres = "".join(ch for ch in s if ch.isdigit())
    return len(chiffres) >= 4  # les codes produit UCI font 5 chiffres (+ suffixe optionnel)

mask_special = ~df_cleaned["StockCode"].apply(est_code_numerique)
codes_speciaux = df_cleaned.loc[mask_special, "StockCode"].value_counts()

print(f"Lignes à code spécial (non-produit) : {mask_special.sum():,} "
      f"({mask_special.mean():.2%} du total)")
display(codes_speciaux.head(20))

Lignes à code spécial (non-produit) : 3,660 (0.46% du total)


StockCode
POST            1983
M               1085
C2               254
D                170
ADJUST            61
BANK CHARGES      37
PADS              19
DOT               16
CRUK              16
TEST001           15
ADJUST2            3
TEST002            1
Name: count, dtype: int64

In [21]:
# Décision explicite et documentée :
# - On CONSERVE ces lignes (elles comptent dans le Montant / la Fréquence du client) ;
# - On les marque avec une colonne booléenne pour pouvoir les exclure d'une analyse
#   "panier produit" plus tard si besoin, sans perdre l'information de dépense ;
# - Seul le sous-ensemble des codes VRAIMENT vides après nettoyage (aucun caractère
#   alphanumérique du tout, ex: valeur corrompue) sera supprimé, pas tout code alphabétique.

df_cleaned["IsCodeSpecial"] = mask_special

def get_stock_code(code):
    """Normalise un StockCode produit en entier. Retourne np.nan si le code
    est un code spécial (traité séparément) ou réellement vide/corrompu.
    Chemin de retour explicite dans tous les cas (plus de retour implicite None)."""
    s = str(code)
    chiffres = "".join(ch for ch in s if ch.isdigit())
    if len(chiffres) == 0:
        return np.nan
    return int(chiffres)

print(get_stock_code("79323W"), "|", get_stock_code("POST"), "|", get_stock_code(85123))

79323 | nan | 85123


In [22]:
df_cleaned["StockCodeNum"] = np.where(
    df_cleaned["IsCodeSpecial"],
    np.nan,
    df_cleaned["StockCode"].apply(get_stock_code),
)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Invoice        797885 non-null  object        
 1   StockCode      797885 non-null  object        
 2   Description    797885 non-null  object        
 3   Quantity       797885 non-null  int64         
 4   InvoiceDate    797885 non-null  datetime64[us]
 5   Price          797885 non-null  float64       
 6   CustomerID     797885 non-null  int64         
 7   Country        797885 non-null  str           
 8   IsCodeSpecial  797885 non-null  bool          
 9   StockCodeNum   794225 non-null  float64       
dtypes: bool(1), datetime64[us](1), float64(2), int64(2), object(3), str(1)
memory usage: 65.7+ MB


In [23]:
# Seules les lignes NI code special NI convertibles en numerique sont
# reellement corrompues -> celles-la seules sont retirees.
corrompues = df_cleaned["StockCodeNum"].isna() & ~df_cleaned["IsCodeSpecial"]
print(f"Lignes réellement corrompues (à supprimer) : {corrompues.sum()}")
display(df_cleaned.loc[corrompues, ["StockCode", "Description"]].head(10))

df_cleaned = df_cleaned.loc[~corrompues].reset_index(drop=True)
df_cleaned.shape

Lignes réellement corrompues (à supprimer) : 0


,StockCode,Description


(797885, 10)

In [24]:
# StockCodeNum reste flottant pour les codes speciaux (NaN) : Int64 nullable
# plutot que int, pour ne pas perdre ces lignes ni forcer une conversion invalide.
df_cleaned["StockCodeNum"] = df_cleaned["StockCodeNum"].astype("Int64")
df_cleaned["Invoice"] = df_cleaned["Invoice"].astype(str)
df_cleaned.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,IsCodeSpecial,StockCodeNum
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,85048
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,79323
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,79323
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,22041
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,21232


### 4. Annulations et valeurs de prix aberrantes  *(nouvelle section — [CORRECTIF])*

Les factures d'annulation (`Invoice` commençant par `'C'`) et les lignes à `Price <= 0` ne
représentent pas des ventes : les inclure gonflerait artificiellement la Fréquence et fausserait
le Montant de la Partie 2 (et ferait planter `log1p` en cas de Montant client négatif).

In [33]:
# Identification explicite des annulations avant de construire la table des ventes.
df_cleaned["IsCancelled"] = df_cleaned["Invoice"].str.startswith("C")
df_cleaned["StockCode"] = df_cleaned["StockCode"].astype("string")

print(f"Lignes d'annulation : {df_cleaned['IsCancelled'].sum():,} "
      f"({df_cleaned['IsCancelled'].mean():.2%})")

# Sauvegarde séparée pour une analyse ultérieure du taux de retour (hors scope RFM de base)
df_cleaned.loc[df_cleaned["IsCancelled"]].to_parquet(
    PROC / "transactions_annulees.parquet", index=False
)

# Les annulations sont exclues du périmètre RFM de base.
ventes = df_cleaned.loc[~df_cleaned["IsCancelled"]].copy()
print("Transactions annulées sauvegardées.")
print(f"Ventes conservées pour le RFM : {len(ventes):,} lignes")

Lignes d'annulation : 18,390 (2.30%)
Transactions annulées sauvegardées.
Ventes conservées pour le RFM : 779,495 lignes


In [30]:
n_avant = len(ventes)
ventes = ventes.loc[ventes["Price"] > 0].copy()
print(f"Lignes avec Price <= 0 supprimées : {n_avant - len(ventes)}")

n_avant = len(ventes)
ventes = ventes.loc[ventes["Quantity"] > 0].copy()
print(f"Lignes avec Quantity <= 0 supprimées (hors annulations déjà retirées) : "
      f"{n_avant - len(ventes)}")

Lignes avec Price <= 0 supprimées : 70
Lignes avec Quantity <= 0 supprimées (hors annulations déjà retirées) : 0


### 5. Calcul du montant et export *(nouveau — [CORRECTIF] : comble le chaînon manquant vers la Partie 2)*

In [31]:
ventes["Amount"] = ventes["Quantity"] * ventes["Price"]

assert ventes["Amount"].min() > 0, "Un Amount <= 0 subsiste : vérifier les filtres ci-dessus."
assert ventes["Invoice"].str.startswith("C").sum() == 0, "Des annulations ont fuité dans 'ventes'."

print("Vérifications passées.")
ventes[["Invoice", "StockCode", "Quantity", "Price", "Amount", "CustomerID"]].describe()

Vérifications passées.


,Quantity,Price,Amount,CustomerID
count,779425.000000,779425.000000,779425.000000,779425.000000
mean,13.489370,3.218488,22.291823,15320.360461
std,145.855814,29.676140,227.427075,1695.692775
min,1.000000,0.001000,0.001000,12346.000000
25%,2.000000,1.250000,4.950000,13971.000000
50%,6.000000,1.950000,12.480000,15247.000000
75%,12.000000,3.750000,19.800000,16794.000000
max,80995.000000,10953.500000,168469.600000,18287.000000


In [32]:
ventes.to_parquet(PROC / "transactions_clean.parquet", index=False)
print("Sauvegardé :", PROC / "transactions_clean.parquet")
print("Shape finale :", ventes.shape)

Sauvegardé : ../data/processed/transactions_clean.parquet
Shape finale : (779425, 12)
